# Feature Representation Analysis: SPO+ vs MSE

This notebook investigates how decision-focused (SPO+) and prediction-focused (MSE) training
produce different learned feature representations in the shared two-layer architecture
(neural feature extractor → per-arm Bayesian linear head).

We use a noiseless custom DGP with a **known oracle decision feature** to isolate the
effect of the loss function on feature learning. The analysis proceeds in two stages:
first under **full feedback** (mechanism isolation), then under **partial feedback**
(semi-bandit degradation).

In [1]:
import sys
sys.path.insert(0, '..')

import numpy as np
from lets_plot import LetsPlot
LetsPlot.setup_html()

from feature_representation_analysis_claude_helpers import (
    # DGP
    DecisionFocusedEnv,
    compute_oracle_decision, decision_label, ORACLE_W1,
    # Training
    create_agent_pair, train_with_checkpoints,
    extract_features, generate_test_contexts, features_at_checkpoint,
    # Alignment
    subspace_angle,
    # Plotting
    plot_feature_grid, plot_feature_directions,
    create_feature_evolution_animation,
    _decision_accuracy, _cost_correlation,
    COLORS, DECISION_COLORS,
)
from src.agents.base import pregenerate_episode

## 1. Custom Data-Generating Process

4-arm unit-weight knapsack (capacity=2). Context $x \in \mathbb{R}^5$.

- **Dims 0–1** (decision): control which items are selected
- **Dims 2–4** (nuisance): shift all costs equally, never flip decisions

The **oracle decision feature** is the projection $x_{0:2}$.

In [2]:
# Create noiseless environment — oracle matches solver exactly
env = DecisionFocusedEnv(seed=42, noise_width=0.0)

# Verify: oracle should match solver on all contexts (no noise)
rng_verify = np.random.default_rng(123)
n_verify = 200
matches = 0
for _ in range(n_verify):
    x = rng_verify.standard_normal(5)
    v = env.data._generate_values(x)
    oracle_sel = compute_oracle_decision(x)
    solver_sel = env.solve(v)
    if np.array_equal(oracle_sel.astype(int), solver_sel.astype(int)):
        matches += 1

print(f"Oracle-solver agreement: {matches}/{n_verify} ({100*matches/n_verify:.0f}%)")

Oracle-solver agreement: 195/200 (98%)


## 2. Train SPO and MSE Agents (Tiny Problem)

Both agents: `hidden_dim=2`, linear features, **full feedback**.
Same pregenerated episode, same initialization.

We use full feedback here to isolate the effect of the loss function on feature learning,
removing the confound of partial observability.

In [3]:
# Noiseless environment for clean mechanism analysis
env = DecisionFocusedEnv(seed=42, noise_width=0.0)
N_ROUNDS = 5000
CHECKPOINT_INTERVAL = 200

# Pregenerate episode
episode = pregenerate_episode(env, n_rounds=N_ROUNDS)

# Create matched agents with aggressive learning parameters:
# - lr=0.03: counters the 1/sqrt(t) decay built into the optimizer
# - lambda_reg=0.1: lower regularization so theta grows faster,
#   giving stronger gradient signal through the Bayesian layer
spo_agent, mse_agent = create_agent_pair(
    env, hidden_dim=2, beta=1.0, lambda_reg=0.1, lr=0.03, seed=0,
)

# Train both on same episode with full feedback
# (isolates loss function effect on feature learning)
res_spo = train_with_checkpoints(
    spo_agent, episode, N_ROUNDS,
    checkpoint_interval=CHECKPOINT_INTERVAL, full_feedback=True,
)
_, mse_agent = create_agent_pair(
    env, hidden_dim=2, beta=1.0, lambda_reg=0.1, lr=0.03, seed=0,
)
res_mse = train_with_checkpoints(
    mse_agent, episode, N_ROUNDS,
    checkpoint_interval=CHECKPOINT_INTERVAL, full_feedback=True,
)

print(f"SPO final cumulative regret: {res_spo['cumulative_regret'][-1]:.1f}")
print(f"MSE final cumulative regret: {res_mse['cumulative_regret'][-1]:.1f}")

SPO final cumulative regret: 1177.9
MSE final cumulative regret: 1225.2


## 3. Feature Space Comparison: Oracle vs SPO vs MSE

**Row 1 — colored by decision:** Which 2-of-4 items are selected. The oracle (left) shows
clean quadrant separation because x₀ and x₁ directly determine the decision.
SPO should look similar; MSE should not.

**Row 2 — colored by total cost magnitude:** A smooth blue→red gradient means the
features organize by "how expensive is this context overall" (nuisance information).
MSE should show this gradient; SPO should not (cost magnitude is decision-irrelevant).

In [4]:
# Generate test contexts (noiseless)
env_test = DecisionFocusedEnv(seed=42, noise_width=0.0)
X_test, decisions_test, values_test = generate_test_contexts(env_test, n_samples=500, seed=99)

# Extract features
h_spo = extract_features(spo_agent, X_test)
h_mse = extract_features(mse_agent, X_test)

# Main figure: 2x3 grid
plot_feature_grid(h_spo, h_mse, X_test, decisions_test, values_test)

/Users/aabhash/Documents/code/DFL for TS Autoregressor/constrained-bandits/.venv/lib/python3.11/site-packages/sklearn/model_selection/_split.py:813: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(
/Users/aabhash/Documents/code/DFL for TS Autoregressor/constrained-bandits/.venv/lib/python3.11/site-packages/sklearn/model_selection/_split.py:813: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(
/Users/aabhash/Documents/code/DFL for TS Autoregressor/constrained-bandits/.venv/lib/python3.11/site-packages/sklearn/model_selection/_split.py:813: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(


## 4. Feature Evolution During Training

Animation showing how the learned feature space develops over 5000 training rounds.
Oracle (left) is static. Watch how SPO clusters gradually separate by decision,
while MSE features organize along a different axis (cost magnitude).

In [5]:
create_feature_evolution_animation(
    res_spo['W1_checkpoints'], res_mse['W1_checkpoints'],
    X_test, decisions_test,
    regrets_spo=res_spo['regrets'],
    regrets_mse=res_mse['regrets'],
    interval_ms=600,
)

## 5. Feature Loading by Context Dimension

How much weight does each agent place on each context dimension?
SPO should load on decision dimensions (x0, x1); MSE should spread more uniformly.

In [6]:
W1_spo_final = res_spo['W1_checkpoints'][-1][1]
W1_mse_final = res_mse['W1_checkpoints'][-1][1]

plot_feature_directions(W1_spo_final, W1_mse_final)

## 6. Decoded Latent Recovery

Can a linear probe reconstruct the true decision latent (x₀) or nuisance latent (x₂) from
the learned 2D features? If features capture a latent, the scatter should follow the diagonal.

- **SPO** should recover x₀ well (decision-relevant) but not x₂ (nuisance)
- **MSE** should recover x₂ well (prediction-relevant) but not x₀

In [7]:
from feature_representation_analysis_claude_helpers import plot_decoded_latents

plot_decoded_latents(h_spo, h_mse, X_test, decisions_test)

## 6. Quantitative Metrics

Three metrics capture different aspects of what the learned features encode:

**Decision classification accuracy.** A multinomial logistic regression is trained to predict
the optimal decision (which 2-of-4 items) from the 2D learned features, using 5-fold
cross-validation. Higher accuracy means the features linearly separate the four decision
regions. A decision-focused loss should produce features where decisions are linearly
decodable; a prediction-focused loss has no such incentive.

**Cost correlation |r|.** The maximum absolute Pearson correlation between either feature
dimension and total cost $\sum_i v_i(x)$. Total cost is driven entirely by the nuisance
dimensions $x_{2:5}$ (which shift all item values equally) and is orthogonal to the
decision-relevant subspace. High |r| means the features encode nuisance structure that
helps predict costs but carries no decision information.

**Subspace alignment angle.** The largest principal angle (in degrees) between the learned
2D feature subspace $\text{colspan}(W_1)$ and the oracle subspace $\text{colspan}([e_0, e_1])$.
Computed via SVD of $Q_1^\top Q_2$ where $Q_1, Q_2$ are QR-orthonormalized bases.
$0°$ = perfect alignment, $90°$ = orthogonal (features ignore decision-relevant dimensions).

In [8]:
oracle_h = X_test[:, :2]
dec_labels = np.array([decision_label(d) for d in decisions_test])
total_cost = values_test.sum(axis=1)

# Compute all metrics
results = {}
for name, h in [('Oracle', oracle_h), ('BayesianSPO', h_spo), ('BayesianMSE', h_mse)]:
    W1 = (ORACLE_W1 if name == 'Oracle'
          else np.array(spo_agent.W1) if name == 'BayesianSPO'
          else np.array(mse_agent.W1))
    results[name] = {
        'Decision accuracy': f"{_decision_accuracy(h, dec_labels):.0%}",
        'Cost correlation |r|': f"{_cost_correlation(h, total_cost):.2f}",
        # 'Subspace angle to oracle': f"{subspace_angle(W1):.1f}\u00b0",
    }

import pandas as pd
df_metrics = pd.DataFrame(results).T
df_metrics.index.name = 'Feature space'
print(df_metrics.to_string())

              Decision accuracy Cost correlation |r|
Feature space                                       
Oracle                      96%                 0.06
BayesianSPO                 72%                 0.53
BayesianMSE                 59%                 1.00


/Users/aabhash/Documents/code/DFL for TS Autoregressor/constrained-bandits/.venv/lib/python3.11/site-packages/sklearn/model_selection/_split.py:813: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(
/Users/aabhash/Documents/code/DFL for TS Autoregressor/constrained-bandits/.venv/lib/python3.11/site-packages/sklearn/model_selection/_split.py:813: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(
/Users/aabhash/Documents/code/DFL for TS Autoregressor/constrained-bandits/.venv/lib/python3.11/site-packages/sklearn/model_selection/_split.py:813: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(


## 7. Partial-Feedback Degradation

The full-feedback analysis isolates the loss function effect. But in the actual semi-bandit
setting, agents only observe costs for selected items (2 of 4). How much does the
SPO advantage degrade?

We train identical agents on the **same episode** with `full_feedback=False` and compare.

In [9]:
# Train under partial (semi-bandit) feedback on the same episode
# SPO uses mask_grad=True (only backprop through observed items)
spo_partial, _ = create_agent_pair(
    env, hidden_dim=2, beta=1.0, lambda_reg=0.1, lr=0.03, seed=0,
    mask_spo=True,
)
res_spo_partial = train_with_checkpoints(
    spo_partial, episode, N_ROUNDS,
    checkpoint_interval=CHECKPOINT_INTERVAL, full_feedback=False,
)
_, mse_partial = create_agent_pair(
    env, hidden_dim=2, beta=1.0, lambda_reg=0.1, lr=0.03, seed=0,
)
res_mse_partial = train_with_checkpoints(
    mse_partial, episode, N_ROUNDS,
    checkpoint_interval=CHECKPOINT_INTERVAL, full_feedback=False,
)

# Extract features and compute metrics
h_spo_p = extract_features(spo_partial, X_test)
h_mse_p = extract_features(mse_partial, X_test)

# Build comparison table: full vs partial
rows_table = {}
for regime, h_s, h_m, agent_s, agent_m, res_s, res_m in [
    ('Full feedback', h_spo, h_mse, spo_agent, mse_agent, res_spo, res_mse),
    ('Partial feedback', h_spo_p, h_mse_p, spo_partial, mse_partial, res_spo_partial, res_mse_partial),
]:
    for name, h, agent, res in [
        ('SPO', h_s, agent_s, res_s),
        ('MSE', h_m, agent_m, res_m),
    ]:
        key = f'{name} ({regime.split()[0].lower()})'
        rows_table[key] = {
            'Decision accuracy': f"{_decision_accuracy(h, dec_labels):.0%}",
            'Cost |r|': f"{_cost_correlation(h, total_cost):.2f}",
            'Alignment': f"{subspace_angle(np.array(agent.W1)):.1f}\u00b0",
            'Regret': f"{res['cumulative_regret'][-1]:.0f}",
        }

df_compare = pd.DataFrame(rows_table).T
df_compare.index.name = 'Agent (regime)'
print(df_compare.to_string())

               Decision accuracy Cost |r| Alignment Regret
Agent (regime)                                            
SPO (full)                   72%     0.53     41.9°   1178
MSE (full)                   59%     1.00     90.0°   1225
SPO (partial)                52%     0.89     79.2°   2603
MSE (partial)                50%     0.84     89.9°   3069


/Users/aabhash/Documents/code/DFL for TS Autoregressor/constrained-bandits/.venv/lib/python3.11/site-packages/sklearn/model_selection/_split.py:813: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(
/Users/aabhash/Documents/code/DFL for TS Autoregressor/constrained-bandits/.venv/lib/python3.11/site-packages/sklearn/model_selection/_split.py:813: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(
/Users/aabhash/Documents/code/DFL for TS Autoregressor/constrained-bandits/.venv/lib/python3.11/site-packages/sklearn/model_selection/_split.py:813: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(
/Users/aabhash/Documents/code/DFL for TS Autoregressor/constrained-bandits/.venv/lib/python3.11/site-packages/sklearn/model_selection/_split.py:813: UserWarning: The least populated class in y has only 2 members, whic

In [10]:
# Feature scatter under partial feedback for visual comparison
plot_feature_grid(h_spo_p, h_mse_p, X_test, decisions_test, values_test)

/Users/aabhash/Documents/code/DFL for TS Autoregressor/constrained-bandits/.venv/lib/python3.11/site-packages/sklearn/model_selection/_split.py:813: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(
/Users/aabhash/Documents/code/DFL for TS Autoregressor/constrained-bandits/.venv/lib/python3.11/site-packages/sklearn/model_selection/_split.py:813: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(
/Users/aabhash/Documents/code/DFL for TS Autoregressor/constrained-bandits/.venv/lib/python3.11/site-packages/sklearn/model_selection/_split.py:813: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(


In [11]:
# Decoded latent recovery under partial feedback
plot_decoded_latents(h_spo_p, h_mse_p, X_test, decisions_test)

In [ ]:
# Feature capacity allocation under partial feedback
W1_spo_p = res_spo_partial['W1_checkpoints'][-1][1]
W1_mse_p = res_mse_partial['W1_checkpoints'][-1][1]
plot_feature_directions(W1_spo_p, W1_mse_p)

In [12]:
create_feature_evolution_animation(
    res_spo_partial['W1_checkpoints'], res_mse_partial['W1_checkpoints'],
    X_test, decisions_test,
    regrets_spo=res_spo_partial['regrets'],
    regrets_mse=res_mse_partial['regrets'],
    interval_ms=600,
)

## Summary

Under **full feedback**, SPO+ features concentrate on the decision-relevant subspace
(high decision accuracy, low cost correlation), while MSE features allocate capacity
to nuisance dimensions that reduce prediction error but carry no decision information.

Under **partial feedback**, both agents degrade — but the comparison table above shows
how much of the SPO advantage survives when only selected items are observed.
See the tables above for exact numbers.